# MoE Prefetch Experiment Analysis
**Date:** 2026-03-24  
**Experiment:** `experiment_20260324_082032`  
**Setup:** GPT-OSS-20B-F16, ctx=131k, 2000 tokens, seed=42, swap OFF  
**Pressures:** 0 GiB (default), 20 GiB, 22 GiB  

**Configs:**
1. Default (out-of-box, prefetch ON, warmup ON) — no pressure, 1 run
2. Lazy (prefetch OFF, no warmup)
3. Lazy + Pinned (mlock attention+output weights)
4. Lazy + Madvise (MADV_WILLNEED prefetch after router)
5. Lazy + Madvise + Pinned

**Note:** `4_lazy_madvise_22g` run 3 is excluded (outlier: 1095s vs 754/613s).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from pathlib import Path

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

RESULT_DIR = Path('results/experiment_20260324_082032')
SSD_CAPACITY_MIB = 7000  # Samsung 980 PRO theoretical max

# Config definitions: (csv_name, display_name, color)
CONFIGS_20G = [
    ('2_lazy_20g',                'Lazy',              '#e67e22'),
    ('3_lazy_pinned_20g',         'Lazy+Pinned',       '#3498db'),
    ('4_lazy_madvise_20g',        'Lazy+Madvise',      '#2ecc71'),
    ('5_lazy_madvise_pinned_20g', 'Lazy+Madvise+Pin',  '#9b59b6'),
]
CONFIGS_22G = [
    ('2_lazy_22g',                'Lazy',              '#e67e22'),
    ('3_lazy_pinned_22g',         'Lazy+Pinned',       '#3498db'),
    ('4_lazy_madvise_22g',        'Lazy+Madvise',      '#2ecc71'),
    ('5_lazy_madvise_pinned_22g', 'Lazy+Madvise+Pin',  '#9b59b6'),
]

# Outlier filter: exclude madvise_22g run 3
EXCLUDE = {('4_lazy_madvise_22g', 3)}

In [ ]:
# Load all timing CSVs
def load_timing(name):
    df = pd.read_csv(RESULT_DIR / f'{name}.csv')
    # Filter outliers
    df = df[~df['run'].apply(lambda r: (name, r) in EXCLUDE)]
    return df

# Load IO CSVs for a config (all iterations)
def load_io(name):
    dfs = {}
    for i in range(1, 4):
        path = RESULT_DIR / f'{name}_io_{i}.csv'
        if path.exists() and (name, i) not in EXCLUDE:
            dfs[i] = pd.read_csv(path)
    return dfs

# Build summary table
def build_summary(configs):
    rows = []
    for csv_name, disp_name, color in configs:
        df = load_timing(csv_name)
        rows.append({
            'Config': disp_name,
            'Runs': len(df),
            'Avg Wall (s)': df['phase_total_wall_ms'].mean() / 1000,
            'Std Wall (s)': df['phase_total_wall_ms'].std() / 1000 if len(df) > 1 else 0,
            'Avg Gen (s)': df['phase_generation_ms'].mean() / 1000,
            'Avg Tok/s': (df['eval_tokens'] / (df['eval_time_ms'] / 1000)).mean(),
            'Avg Faults (M)': df['phase_total_faults'].mean() / 1e6,
            'Avg Gen Faults (M)': df['phase_generation_faults'].mean() / 1e6,
            'Avg Prompt Faults': df['phase_prompt_eval_faults'].mean(),
        })
    summary = pd.DataFrame(rows)
    # Add relative columns
    baseline_wall = summary.iloc[0]['Avg Wall (s)']
    summary['vs Lazy'] = ((summary['Avg Wall (s)'] - baseline_wall) / baseline_wall * 100).round(1)
    return summary

print('=== 20 GiB Pressure ===')
s20 = build_summary(CONFIGS_20G)
print(s20.to_string(index=False))

print('\n=== 22 GiB Pressure ===')
s22 = build_summary(CONFIGS_22G)
print(s22.to_string(index=False))

## 1. Summary Bar Chart — Wall Time by Config

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, configs, pressure, summary in [
    (axes[0], CONFIGS_20G, '20 GiB', s20),
    (axes[1], CONFIGS_22G, '22 GiB', s22),
]:
    names = [c[1] for c in configs]
    colors = [c[2] for c in configs]
    walls = summary['Avg Wall (s)'].values
    stds = summary['Std Wall (s)'].values
    
    bars = ax.bar(names, walls, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.errorbar(names, walls, yerr=stds, fmt='none', ecolor='black', capsize=5, linewidth=1.5)
    
    # Annotate with wall time and % vs lazy
    for i, (bar, wall, vs) in enumerate(zip(bars, walls, summary['vs Lazy'].values)):
        label = f'{wall:.0f}s'
        if i > 0:
            label += f'\n({vs:+.1f}%)'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + stds[i] + 5,
                label, ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_title(f'{pressure} Pressure', fontsize=14, fontweight='bold')
    ax.set_ylabel('Wall Time (seconds)')
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

fig.suptitle('GPT-OSS-20B — 2000 Tokens, ctx=131k, swap OFF, seed=42',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('io_plots/summary_wall_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Page Faults Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, configs, pressure, summary in [
    (axes[0], CONFIGS_20G, '20 GiB', s20),
    (axes[1], CONFIGS_22G, '22 GiB', s22),
]:
    names = [c[1] for c in configs]
    colors = [c[2] for c in configs]
    gen_faults = summary['Avg Gen Faults (M)'].values
    prompt_faults = summary['Avg Prompt Faults'].values / 1e6
    
    bars_gen = ax.bar(names, gen_faults, color=colors, alpha=0.85,
                      edgecolor='black', linewidth=0.5, label='Generation')
    bars_prompt = ax.bar(names, prompt_faults, bottom=gen_faults, color=colors,
                         alpha=0.4, edgecolor='black', linewidth=0.5, label='Prompt eval')
    
    for bar, total in zip(bars_gen, summary['Avg Faults (M)'].values):
        ax.text(bar.get_x() + bar.get_width()/2, total + 0.1,
                f'{total:.1f}M', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_title(f'{pressure} Pressure', fontsize=14, fontweight='bold')
    ax.set_ylabel('Major Page Faults (millions)')
    ax.legend(loc='upper right')

fig.suptitle('Page Faults by Config — Generation vs Prompt Eval',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('io_plots/summary_page_faults.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. SSD Throughput Timelines — 4-Panel Grid (22 GiB)

In [ ]:
def plot_throughput_grid(configs, pressure_label, run=1):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    axes = axes.flatten()
    
    for ax, (csv_name, display, color) in zip(axes, configs):
        io_dfs = load_io(csv_name)
        if run not in io_dfs:
            ax.text(0.5, 0.5, 'No IO data', transform=ax.transAxes, ha='center')
            ax.set_title(display)
            continue
        
        io = io_dfs[run]
        t = io['timestamp_s']
        r = io['read_mib_s']
        
        ax.fill_between(t, r, alpha=0.3, color=color)
        ax.plot(t, r, alpha=0.6, color=color, linewidth=0.3)
        
        # Stats
        mean_r = r[r > 10].mean()  # exclude near-zero samples
        total_read_gib = (io['read_bytes'].sum()) / (1024**3)
        duration = t.iloc[-1]
        
        ax.axhline(mean_r, color='black', linestyle='--', alpha=0.5, linewidth=1)
        ax.axhline(SSD_CAPACITY_MIB, color='red', linestyle='--', alpha=0.3, linewidth=1)
        
        ax.set_title(f'{display} — {duration:.0f}s, {total_read_gib:.0f} GiB read, '
                     f'mean {mean_r:.0f} MiB/s ({mean_r/SSD_CAPACITY_MIB*100:.0f}%)',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (seconds)')
        ax.set_ylabel('Read MiB/s')
        ax.set_ylim(0, SSD_CAPACITY_MIB * 1.05)
    
    fig.suptitle(f'SSD Read Throughput — {pressure_label} pressure, swap OFF, ctx=131k, 2000 tokens (run {run})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig

fig = plot_throughput_grid(CONFIGS_22G, '22 GiB', run=1)
fig.savefig('io_plots/throughput_timeline_22g.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SSD Throughput Timelines — 4-Panel Grid (20 GiB)

In [ ]:
fig = plot_throughput_grid(CONFIGS_20G, '20 GiB', run=1)
fig.savefig('io_plots/throughput_timeline_20g.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Throughput Histograms — All Configs (22 GiB)

In [ ]:
def plot_throughput_histogram(configs, pressure_label, run=1):
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for csv_name, display, color in configs:
        io_dfs = load_io(csv_name)
        if run not in io_dfs:
            continue
        io = io_dfs[run]
        r = io['read_mib_s']
        r_active = r[r > 100]  # filter idle samples
        mean_r = r_active.mean()
        ax.hist(r_active, bins=80, alpha=0.4, color=color,
                label=f'{display} (mean: {mean_r:.0f} MiB/s)', edgecolor=color, linewidth=0.5)
    
    ax.axvline(SSD_CAPACITY_MIB, color='red', linestyle='--', linewidth=2, label=f'SSD capacity ({SSD_CAPACITY_MIB} MiB/s)')
    ax.set_xlabel('Read Throughput (MiB/s)')
    ax.set_ylabel('Count (samples)')
    ax.set_title(f'Throughput Distribution — {pressure_label} pressure (run {run})', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    return fig

fig = plot_throughput_histogram(CONFIGS_22G, '22 GiB', run=1)
fig.savefig('io_plots/throughput_histogram_22g.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. First 30 Seconds Zoom — Startup Behavior (22 GiB)

In [ ]:
def plot_first_n_seconds(configs, pressure_label, n_seconds=30, run=1):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    axes = axes.flatten()
    
    for ax, (csv_name, display, color) in zip(axes, configs):
        io_dfs = load_io(csv_name)
        if run not in io_dfs:
            ax.text(0.5, 0.5, 'No IO data', transform=ax.transAxes, ha='center')
            ax.set_title(display)
            continue
        
        io = io_dfs[run]
        mask = io['timestamp_s'] <= n_seconds
        t = io.loc[mask, 'timestamp_s']
        r = io.loc[mask, 'read_mib_s']
        
        ax.plot(t, r, alpha=0.8, color=color, linewidth=0.5)
        ax.fill_between(t, r, alpha=0.2, color=color)
        ax.set_title(f'{display} — first {n_seconds}s', fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (seconds)')
        ax.set_ylabel('Read MiB/s')
        ax.set_ylim(0, 6000)
    
    fig.suptitle(f'Startup Behavior — {pressure_label} pressure, first {n_seconds}s (run {run})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig

fig = plot_first_n_seconds(CONFIGS_22G, '22 GiB', n_seconds=30, run=1)
fig.savefig('io_plots/startup_30s_22g.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Tok/s Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, configs, pressure, summary in [
    (axes[0], CONFIGS_20G, '20 GiB', s20),
    (axes[1], CONFIGS_22G, '22 GiB', s22),
]:
    names = [c[1] for c in configs]
    colors = [c[2] for c in configs]
    toks = summary['Avg Tok/s'].values
    
    bars = ax.bar(names, toks, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, tok in zip(bars, toks):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{tok:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_title(f'{pressure} Pressure', fontsize=14, fontweight='bold')
    ax.set_ylabel('Tokens/second')
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

# Add default (no pressure) reference line
default_df = load_timing('1_default_no_pressure')
default_toks = (default_df['eval_tokens'] / (default_df['eval_time_ms'] / 1000)).mean()
for ax in axes:
    ax.axhline(default_toks, color='red', linestyle='--', alpha=0.5, linewidth=1.5,
               label=f'Default no pressure: {default_toks:.1f} tok/s')
    ax.legend(loc='upper right')

fig.suptitle('Generation Throughput — Tokens/second', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('io_plots/summary_tok_per_sec.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Raw Data Inspection

In [ ]:
# Print all raw timing data for reference
all_configs = [('1_default_no_pressure', 'Default (0 GiB)')] + \
    [(c[0], f'{c[1]} (20g)') for c in CONFIGS_20G] + \
    [(c[0], f'{c[1]} (22g)') for c in CONFIGS_22G]

for csv_name, disp_name in all_configs:
    df = load_timing(csv_name)
    print(f'\n--- {disp_name} ({len(df)} runs) ---')
    cols = ['run', 'phase_total_wall_ms', 'phase_generation_ms', 'phase_prompt_eval_ms',
            'phase_total_faults', 'phase_generation_faults', 'mlock_mib']
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))